In [1]:
import torch
from transformers import AutoProcessor, LlavaForConditionalGeneration

model_id = "llava-hf/llava-1.5-7b-hf"

processor = AutoProcessor.from_pretrained(model_id)
print("Processor loaded successfully!")

model = LlavaForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="cuda",
    low_cpu_mem_usage=True
)

model.eval()
print("Model loaded successfully!")

/home/phuong/anaconda3/envs/vlm-llava/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Processor loaded successfully!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]/home/phuong/anaconda3/envs/vlm-llava/lib/python3.12/site-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 NVIDIA GB10 which is of cuda capability 12.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (8.0) - (12.0)
    
  queued_call()
Loading checkpoint shards: 100%|██████████| 3/3 [03:18<00:00, 66.12s/it]

Model loaded successfully!


In [ ]:
# ============ COCO VAL2017 DoLa-low Inference ============
import json
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import torch

COCO_IMG_DIR = "../data/coco2017/val2017"
COCO_ANN_PATH = "../data/coco2017/annotations/instances_val2017.json"
OUTPUT_PATH = "../results/infer/coco_val2017/llava15/dola_low.json"

BATCH_SIZE = 16
MAX_NEW_TOKENS = 256
PROMPT = "Describe this image."

# DoLa config
DOLA_LAYERS = "low"
REPETITION_PENALTY = 1.2

processor.tokenizer.padding_side = "left"
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

model.generation_config.pad_token_id = processor.tokenizer.pad_token_id
model.generation_config.eos_token_id = processor.tokenizer.eos_token_id


def batch_list(items, batch_size):
    for i in range(0, len(items), batch_size):
        yield items[i:i + batch_size]


def load_image(path):
    return Image.open(path).convert("RGB")


def clean_output(text):
    text = text.strip()
    if "ASSISTANT:" in text:
        text = text.split("ASSISTANT:", 1)[-1].strip()
    return text


# -----------------------------
# Load COCO val2017 annotation
# -----------------------------
with open(COCO_ANN_PATH, "r", encoding="utf-8") as f:
    coco = json.load(f)

images_info = sorted(coco["images"], key=lambda x: x["id"])

print(f"Loaded {len(images_info)} COCO val2017 images.")
print("Example:", images_info[0])

# -----------------------------
# Run batched DoLa-low inference
# -----------------------------
results = []
hf_prompt = f"USER: <image>\n{PROMPT} ASSISTANT:"

Path(OUTPUT_PATH).parent.mkdir(parents=True, exist_ok=True)

num_batches = (len(images_info) + BATCH_SIZE - 1) // BATCH_SIZE

for batch_idx, batch in enumerate(tqdm(batch_list(images_info, BATCH_SIZE), total=num_batches)):
    batch_images = []
    batch_prompts = []
    batch_ids = []

    # only preparing one mini-batch here
    for item in batch:
        image_id = int(item["id"])
        file_name = item["file_name"]

        image_path = Path(COCO_IMG_DIR) / file_name
        if not image_path.exists():
            raise FileNotFoundError(f"Missing image: {image_path}")

        image = load_image(image_path)

        batch_images.append(image)
        batch_prompts.append(hf_prompt)
        batch_ids.append(image_id)

    inputs = processor(
        text=batch_prompts,
        images=batch_images,
        return_tensors="pt",
        padding=True
    )

    input_device = next(model.parameters()).device
    inputs = {
        k: v.to(input_device) if isinstance(v, torch.Tensor) else v
        for k, v in inputs.items()
    }

    input_len = inputs["input_ids"].shape[1]

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            use_cache=True,
            pad_token_id=processor.tokenizer.pad_token_id,
            eos_token_id=processor.tokenizer.eos_token_id,
            dola_layers=DOLA_LAYERS,              # DoLa-low
            repetition_penalty=REPETITION_PENALTY
        )

    generated_ids = output_ids[:, input_len:]

    outputs = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True
    )

    for image_id, output in zip(batch_ids, outputs):
        caption = clean_output(output)
        results.append({
            "id": image_id,
            "caption": caption
        })

# -----------------------------
# Save results
# -----------------------------
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"\nSaved {len(results)} captions to {OUTPUT_PATH}")

Loaded 5000 COCO val2017 images.
Example: {'license': 2, 'file_name': '000000000139.jpg', 'coco_url': 'http://images.cocodataset.org/val2017/000000000139.jpg', 'height': 426, 'width': 640, 'date_captured': '2013-11-21 01:34:01', 'flickr_url': 'http://farm9.staticflickr.com/8035/8024364858_9c41dc1666_z.jpg', 'id': 139}


 25%|██▍       | 77/313 [1:29:22<3:03:15, 46.59s/it]